# G1 Platform MVP — Colab GPU runner

Runs the Phase 0/1 MVP (vLLM + FastAPI gateway + LangGraph agent + load
simulator) on Colab's free GPU, without Docker and without needing a
persistent SSH box.

**Before running anything:** go to `Runtime -> Change runtime type` and
set Hardware accelerator to **T4 GPU** (or better, if you have Colab
Pro). Then run the cells top to bottom.

Notes on the free tier:
- The GPU you get is whatever's free right now (usually a T4, 16GB VRAM)
  — availability isn't guaranteed and Colab can disconnect an idle
  session after ~90 minutes.
- This notebook defaults to `Qwen2.5-3B-Instruct`, which fits a T4
  comfortably. If you hit an out-of-memory error, switch `VLLM_MODEL`
  below to `Qwen/Qwen2.5-1.5B-Instruct` and re-run from that cell down.


## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi


## 2. Install dependencies

This takes a few minutes the first time (vLLM pulls in a lot).

In [ ]:
!pip install -q vllm fastapi "uvicorn[standard]" httpx pydantic langgraph langchain-core python-dotenv


## 3. Write out the project files

Same code as the `g1-mvp` scaffold — just materialized directly in the Colab filesystem so there's nothing to upload.

In [ ]:
import os
os.makedirs('g1-mvp/agent', exist_ok=True)
os.makedirs('g1-mvp/gateway', exist_ok=True)
os.makedirs('g1-mvp/simulator', exist_ok=True)
print('directories ready')


In [ ]:
%%writefile g1-mvp/agent/__init__.py


In [ ]:
%%writefile g1-mvp/gateway/__init__.py


In [ ]:
%%writefile g1-mvp/simulator/__init__.py


In [ ]:
%%writefile g1-mvp/agent/tools.py
"""
Phase 0 tools for the G1 orchestration graph.

Per the technical proposal (Section 5.1, Phase 0 — Proof of Concept), the
goal here is to prove the *shape* of the pipeline — a tool call that feeds
context into the model call — before building the real GPU-metrics
(nvidia-smi) and FAISS retrieval tools called for in Phase 2.

This module intentionally ships a single fake tool, `fake_context_tool`,
that returns a canned but plausible-looking payload. Swap its body for a
real `nvidia-smi` read or a FAISS similarity search when you move into
Phase 2 — the call signature (`goal: str -> dict`) is designed to stay
the same so the graph in `agent/graph.py` does not need to change.
"""

from __future__ import annotations

import time


def fake_context_tool(goal: str) -> dict:
    """
    Stand-in for the real tools (GPU metrics / retrieval) described in the
    proposal. Returns deterministic, inspectable "context" so you can see,
    end to end, that the tool's output actually reaches the final prompt.

    Replace this with:
      - a live `nvidia-smi --query-gpu=...` read (Phase 2 GPU-metrics tool), or
      - a FAISS similarity search against an embedded knowledge base
        (Phase 2 retrieval tool).
    """
    time.sleep(0.05)  # simulate a small amount of real tool latency
    return {
        "source": "fake_context_tool",
        "goal_received": goal,
        "note": (
            "This is placeholder context from Phase 0. In Phase 2 this "
            "will be replaced by a live nvidia-smi read and/or a FAISS "
            "retrieval hit against a real knowledge base."
        ),
        "sample_metric": {"gpu_util_pct": 42.0, "mem_used_mb": 8192},
    }


In [ ]:
%%writefile g1-mvp/agent/graph.py
"""
Two-node LangGraph agent: tool node -> model node.

This is the graph called for by Section 5.4 (Definition of Done): "The
response is produced by a compiled LangGraph agent that calls at least one
tool" — not a linear if/else script. Phase 0 uses `fake_context_tool`;
Phase 2 swaps in the real GPU-metrics and FAISS tools without touching the
graph shape.

Failure handling (Section 7 of the proposal):
  - Tool call hangs -> explicit timeout, falls back to inference-only.
  - vLLM unreachable / errors -> caught and surfaced as a clear error
    message rather than a raw stack trace, so the gateway can decide how
    to respond to the caller.
"""

from __future__ import annotations

import asyncio
import os
import time
from typing import Optional, TypedDict

import httpx
from langgraph.graph import StateGraph, END

from agent.tools import fake_context_tool

VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_MODEL = os.getenv("VLLM_MODEL", "Qwen/Qwen2.5-3B-Instruct")
TOOL_TIMEOUT_SECONDS = float(os.getenv("TOOL_TIMEOUT_SECONDS", "3"))


class AgentState(TypedDict, total=False):
    task_id: str
    goal: str
    max_tokens: int
    temperature: float
    tool_context: Optional[dict]
    tool_error: Optional[str]
    answer: Optional[str]
    model_error: Optional[str]
    timings_ms: dict


def _now_ms() -> float:
    return time.perf_counter() * 1000


async def tool_node(state: AgentState) -> AgentState:
    """Calls the (currently fake) context tool with a hard timeout."""
    t0 = _now_ms()
    goal = state["goal"]
    try:
        # fake_context_tool is synchronous/cheap; run it in a thread so a
        # slower real tool (nvidia-smi subprocess, FAISS query) can later
        # drop in here without blocking the event loop.
        result = await asyncio.wait_for(
            asyncio.to_thread(fake_context_tool, goal),
            timeout=TOOL_TIMEOUT_SECONDS,
        )
        state["tool_context"] = result
    except asyncio.TimeoutError:
        state["tool_context"] = None
        state["tool_error"] = f"tool call exceeded {TOOL_TIMEOUT_SECONDS}s timeout"
    except Exception as exc:  # pragma: no cover - defensive
        state["tool_context"] = None
        state["tool_error"] = f"tool call failed: {exc}"
    state.setdefault("timings_ms", {})["tool_node"] = round(_now_ms() - t0, 2)
    return state


async def model_node(state: AgentState) -> AgentState:
    """Calls the vLLM OpenAI-compatible chat completions endpoint."""
    t0 = _now_ms()
    goal = state["goal"]
    tool_context = state.get("tool_context")
    tool_error = state.get("tool_error")

    system_parts = [
        "You are the G1 platform's assistant. Ground your answer in the "
        "tool context provided below when it is relevant."
    ]
    if tool_context:
        system_parts.append(f"Tool context: {tool_context}")
    elif tool_error:
        system_parts.append(
            f"Note: the context tool failed ({tool_error}); answer from "
            "the goal alone and say you had no tool context."
        )

    payload = {
        "model": VLLM_MODEL,
        "messages": [
            {"role": "system", "content": "\n".join(system_parts)},
            {"role": "user", "content": goal},
        ],
        "max_tokens": state.get("max_tokens", 256),
        "temperature": state.get("temperature", 0.7),
    }

    try:
        async with httpx.AsyncClient(timeout=60.0) as client:
            resp = await client.post(
                f"{VLLM_BASE_URL}/chat/completions", json=payload
            )
            resp.raise_for_status()
            data = resp.json()
            state["answer"] = data["choices"][0]["message"]["content"]
    except Exception as exc:
        state["model_error"] = str(exc)
        state["answer"] = None

    state.setdefault("timings_ms", {})["model_node"] = round(_now_ms() - t0, 2)
    return state


def build_graph():
    graph = StateGraph(AgentState)
    graph.add_node("tool_node", tool_node)
    graph.add_node("model_node", model_node)
    graph.set_entry_point("tool_node")
    graph.add_edge("tool_node", "model_node")
    graph.add_edge("model_node", END)
    return graph.compile()


# Compiled once at import time and reused across requests.
compiled_graph = build_graph()


async def run_agent(
    task_id: str,
    goal: str,
    max_tokens: int = 256,
    temperature: float = 0.7,
) -> AgentState:
    initial_state: AgentState = {
        "task_id": task_id,
        "goal": goal,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    return await compiled_graph.ainvoke(initial_state)


In [ ]:
%%writefile g1-mvp/gateway/main.py
"""
FastAPI gateway for the G1 CUDA-Accelerated LLM Inference & Agent
Orchestration Platform (MVP / Phase 0-1 slice).

Exposes:
  POST /v1/agent/invoke  - the agent task request contract from
                            Section 3.3 of the technical proposal.
  GET  /health            - readiness probe; used by Compose's health
                            check and by the gateway's own startup gate
                            (Section 7 risk: "model fails to load at
                            startup").

Run directly with:
    uvicorn gateway.main:app --host 0.0.0.0 --port 8080
"""

from __future__ import annotations

import os
import time
import uuid
from typing import Any, Optional

import httpx
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from agent.graph import VLLM_BASE_URL, run_agent

GATEWAY_HOST = os.getenv("GATEWAY_HOST", "0.0.0.0")
GATEWAY_PORT = int(os.getenv("GATEWAY_PORT", "8080"))

app = FastAPI(
    title="G1 Agent Orchestration Gateway",
    description="MVP gateway: validates requests, runs the LangGraph agent, "
    "returns a grounded response.",
    version="0.1.0",
)


class AgentTaskRequest(BaseModel):
    """Matches Section 3.3 'Agent task request (gateway -> orchestration layer)'."""

    task_id: Optional[str] = Field(default=None, description="Client-supplied trace id; generated if omitted.")
    goal: str = Field(..., min_length=1, description="Natural-language objective the agent must act on.")
    context: Optional[dict[str, Any]] = Field(default=None, description="Conversation id / prior turns, if any.")
    allowed_tools: Optional[list[str]] = Field(default=None, description="Tools the orchestration layer may call.")
    max_tokens: int = Field(default=256, ge=1, le=4096)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)


class AgentTaskResponse(BaseModel):
    task_id: str
    answer: Optional[str]
    tool_context: Optional[dict[str, Any]]
    tool_error: Optional[str]
    model_error: Optional[str]
    latency_ms: float
    timings_ms: dict[str, float]


@app.get("/health")
async def health():
    """
    Reports the gateway's own status plus whether it can currently reach
    the vLLM server. Compose's health check and the gateway's startup
    gate (Section 7: "model fails to load at startup") both key off this.
    """
    vllm_ok = False
    detail = None
    try:
        async with httpx.AsyncClient(timeout=2.0) as client:
            resp = await client.get(f"{VLLM_BASE_URL}/models")
            vllm_ok = resp.status_code == 200
    except Exception as exc:
        detail = str(exc)

    return {
        "gateway": "ok",
        "vllm_reachable": vllm_ok,
        "vllm_base_url": VLLM_BASE_URL,
        "detail": detail,
    }


@app.post("/v1/agent/invoke", response_model=AgentTaskResponse)
async def invoke_agent(request: AgentTaskRequest):
    task_id = request.task_id or str(uuid.uuid4())
    t0 = time.perf_counter()

    try:
        result = await run_agent(
            task_id=task_id,
            goal=request.goal,
            max_tokens=request.max_tokens,
            temperature=request.temperature,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"agent run failed: {exc}") from exc

    latency_ms = round((time.perf_counter() - t0) * 1000, 2)

    return AgentTaskResponse(
        task_id=task_id,
        answer=result.get("answer"),
        tool_context=result.get("tool_context"),
        tool_error=result.get("tool_error"),
        model_error=result.get("model_error"),
        latency_ms=latency_ms,
        timings_ms=result.get("timings_ms", {}),
    )


if __name__ == "__main__":
    import uvicorn

    uvicorn.run("gateway.main:app", host=GATEWAY_HOST, port=GATEWAY_PORT, reload=False)


In [ ]:
%%writefile g1-mvp/simulator/load_test.py
"""
Concurrent request simulator for the G1 platform (Section 8: Validation,
Metrics & Definition of Done).

Runs the gateway at concurrency levels 1, 5, 10, and 20 (configurable),
and reports/saves, per level:
  - p50 and p95 end-to-end latency (ms)
  - aggregate tokens/sec (approximated from response length; see note
    below — wire in real vLLM usage stats for an exact number)
  - success vs failure count

Usage:
    python -m simulator.load_test --url http://localhost:8080/v1/agent/invoke \
        --levels 1 5 10 20 --requests-per-level 10 \
        --goal "Summarize current GPU utilization and recommend an action."

Output:
    Prints a results table to stdout and writes simulator/results.csv.
"""

from __future__ import annotations

import argparse
import asyncio
import csv
import statistics
import time
from pathlib import Path

import httpx

DEFAULT_GOAL = "Summarize current GPU utilization and recommend an action."


async def _one_request(client: httpx.AsyncClient, url: str, goal: str) -> dict:
    t0 = time.perf_counter()
    ok = True
    tokens_est = 0
    try:
        resp = await client.post(url, json={"goal": goal}, timeout=60.0)
        resp.raise_for_status()
        data = resp.json()
        answer = data.get("answer") or ""
        # Rough token estimate (words * 1.3) purely for a throughput signal
        # in the MVP. Replace with real usage.total_tokens from the vLLM
        # response once the gateway passes that field through.
        tokens_est = max(1, int(len(answer.split()) * 1.3))
    except Exception:
        ok = False
    latency_ms = (time.perf_counter() - t0) * 1000
    return {"ok": ok, "latency_ms": latency_ms, "tokens_est": tokens_est}


async def _run_level(url: str, goal: str, concurrency: int, requests_per_level: int) -> dict:
    total_requests = concurrency * requests_per_level
    async with httpx.AsyncClient() as client:
        sem = asyncio.Semaphore(concurrency)

        async def bound():
            async with sem:
                return await _one_request(client, url, goal)

        t0 = time.perf_counter()
        results = await asyncio.gather(*[bound() for _ in range(total_requests)])
        wall_s = time.perf_counter() - t0

    latencies = sorted(r["latency_ms"] for r in results if r["ok"])
    failures = sum(1 for r in results if not r["ok"])
    total_tokens = sum(r["tokens_est"] for r in results if r["ok"])

    def pctile(data, p):
        if not data:
            return float("nan")
        k = (len(data) - 1) * p
        f, c = int(k), min(int(k) + 1, len(data) - 1)
        return data[f] + (data[c] - data[f]) * (k - f)

    return {
        "concurrency": concurrency,
        "total_requests": total_requests,
        "failures": failures,
        "p50_ms": round(pctile(latencies, 0.50), 1),
        "p95_ms": round(pctile(latencies, 0.95), 1),
        "tokens_per_sec": round(total_tokens / wall_s, 1) if wall_s > 0 else 0.0,
        "wall_s": round(wall_s, 2),
    }


async def main_async(args):
    rows = []
    for level in args.levels:
        print(f"Running concurrency={level} ...")
        row = await _run_level(args.url, args.goal, level, args.requests_per_level)
        rows.append(row)
        print(
            f"  p50={row['p50_ms']}ms  p95={row['p95_ms']}ms  "
            f"tok/s~={row['tokens_per_sec']}  failures={row['failures']}/{row['total_requests']}"
        )

    out_path = Path(__file__).parent / "results.csv"
    with out_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"\nSaved results to {out_path}")


def main():
    parser = argparse.ArgumentParser(description="G1 MVP load simulator")
    parser.add_argument("--url", default="http://localhost:8080/v1/agent/invoke")
    parser.add_argument("--levels", nargs="+", type=int, default=[1, 5, 10, 20])
    parser.add_argument("--requests-per-level", type=int, default=10)
    parser.add_argument("--goal", default=DEFAULT_GOAL)
    args = parser.parse_args()
    asyncio.run(main_async(args))


if __name__ == "__main__":
    main()


## 4. Start vLLM in the background

Launches the OpenAI-compatible server as a background process and logs to
`vllm.log`. First run downloads the model weights (a few GB) — this cell
returns immediately; the wait loop in the next cell is what actually
blocks until the server is ready.

If this OOMs on your GPU, change `VLLM_MODEL` here to
`Qwen/Qwen2.5-1.5B-Instruct` and re-run from this cell down.

In [ ]:
import subprocess, os

VLLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # fallback: "Qwen/Qwen2.5-1.5B-Instruct"
os.environ["VLLM_MODEL"] = VLLM_MODEL

vllm_log = open("vllm.log", "w")
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", VLLM_MODEL,
        "--gpu-memory-utilization", "0.85",
        "--max-model-len", "4096",
    ],
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
)
print(f"vLLM starting (pid={vllm_proc.pid}), logging to vllm.log")


In [ ]:
import time, httpx

print("Waiting for vLLM to report ready (this can take a few minutes on first run)...")
for attempt in range(180):
    try:
        r = httpx.get("http://localhost:8000/v1/models", timeout=2.0)
        if r.status_code == 200:
            print("vLLM is up:", r.json())
            break
    except Exception:
        pass
    time.sleep(5)
else:
    print("vLLM did not come up in time — check vllm.log below for the real error.")

!tail -n 40 vllm.log


## 5. Start the FastAPI gateway in the background

In [ ]:
import subprocess, os

gw_env = os.environ.copy()
gw_env["VLLM_BASE_URL"] = "http://localhost:8000/v1"
gw_env["VLLM_MODEL"] = VLLM_MODEL
gw_env["GATEWAY_HOST"] = "0.0.0.0"
gw_env["GATEWAY_PORT"] = "8080"
gw_env["TOOL_TIMEOUT_SECONDS"] = "3"

gw_log = open("gateway.log", "w")
gateway_proc = subprocess.Popen(
    ["uvicorn", "gateway.main:app", "--host", "0.0.0.0", "--port", "8080"],
    cwd="g1-mvp",
    env=gw_env,
    stdout=gw_log,
    stderr=subprocess.STDOUT,
)
print(f"Gateway starting (pid={gateway_proc.pid}), logging to gateway.log")

time.sleep(5)
!tail -n 40 gateway.log


## 6. Smoke test

Equivalent to `scripts/smoke_test.sh` — direct vLLM call, gateway health check, then a full agent invoke.

In [ ]:
import httpx, json

print("== vLLM direct ==")
r = httpx.get("http://localhost:8000/v1/models", timeout=10)
print(json.dumps(r.json(), indent=2))

print("\n== Gateway health ==")
r = httpx.get("http://localhost:8080/health", timeout=10)
print(json.dumps(r.json(), indent=2))

print("\n== Gateway agent invoke ==")
r = httpx.post(
    "http://localhost:8080/v1/agent/invoke",
    json={"goal": "Summarize current GPU utilization and recommend an action."},
    timeout=60,
)
print(json.dumps(r.json(), indent=2))


## 7. Load simulator — the real measured numbers

Runs the concurrency sweep (1/5/10/20) against the gateway and saves `g1-mvp/simulator/results.csv`.

In [ ]:
%cd g1-mvp
!python -m simulator.load_test --levels 1 5 10 20 --requests-per-level 10
%cd ..

import pandas as pd
df = pd.read_csv("g1-mvp/simulator/results.csv")
df


## 8. When you're done

Colab GPUs are a shared, time-limited resource — free the runtime when
you're finished so you're not holding a GPU idle:

`Runtime -> Disconnect and delete runtime`

To download your results (e.g. `results.csv`, `vllm.log`) before
disconnecting, use the Files panel on the left, or:
```
from google.colab import files
files.download('g1-mvp/simulator/results.csv')
```
